<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# Wildfire risk score per building

A raster + vector fusion workflow. We answer:

> **Given a county's terrain steepness, fuel load, building footprints, and road network, score every building for wildfire risk weighted by distance from the nearest evacuation route.**

This is the workflow GeoPandas alone can't do — it mixes raster algebra, raster→vector aggregation, and vector-on-vector distance joins in one pipeline. Sedona makes it one SQL block.

**Pipeline:**
1. Synthesize a slope raster and a fuel-load raster.
2. Combine into a composite risk raster with two-raster `RS_MapAlgebra`.
3. Synthesize building footprints (4×4 grid of polygons) and a small road network.
4. Score each building with `RS_ZonalStats(composite, footprint, 'mean')` × evacuation factor from `ST_DistanceSpheroid` to the nearest road.
5. Rank, write top-5 as GeoParquet, plot.

All inputs are synthesized in the notebook so it's fully reproducible and ships no new bytes. Swap the synthesis cell for `sedona.read.format("raster").load("...")` over a real DEM-derived slope raster and a NLCD-derived fuel raster; everything below is unchanged.

**Requires Sedona ≥ 1.9.0** for the auto-tiling raster reader and proj4sedona-backed `ST_Transform`.

## 1. Connect to Spark through SedonaContext

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

## 2. Synthesize the raster inputs

Two 256×256 single-band rasters over a 0.1° AOI in eastern Iowa:

- `slope.tif` — normalized 0-1, highest along the eastern edge (think: foothills rising toward the east).
- `fuel.tif` — normalized 0-1, highest along the northern edge (think: dense vegetation belt across the top of the AOI).

Both written as **tiled** GeoTIFFs so the Sedona raster reader (which rejects strip-based files as "too thin") can ingest them.

In [ ]:
import os

import numpy as np
import rasterio
from rasterio.transform import from_bounds

WORK = "/tmp/fire-risk"
os.makedirs(WORK, exist_ok=True)

AOI = (-91.10, 41.50, -91.00, 41.60)  # (xmin, ymin, xmax, ymax) EPSG:4326
W = H = 256
transform = from_bounds(*AOI, W, H)
rng = np.random.default_rng(7)

ys, xs = np.mgrid[0:H, 0:W]
slope = (xs / (W - 1)) + 0.05 * rng.standard_normal((H, W))
fuel = ((H - 1 - ys) / (H - 1)) + 0.05 * rng.standard_normal((H, W))
slope = slope.clip(0, 1).astype("float32")
fuel = fuel.clip(0, 1).astype("float32")

for name, arr in [("slope", slope), ("fuel", fuel)]:
    with rasterio.open(
        f"{WORK}/{name}.tif",
        "w",
        driver="GTiff",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        height=H,
        width=W,
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(arr, 1)
        dst.set_band_description(1, name)
    print(f"{name}.tif: min={arr.min():.2f} max={arr.max():.2f} mean={arr.mean():.2f}")

## 3. Combine slope and fuel into a composite risk raster

Load both rasters with the auto-tiling reader, keep the `(x, y)` tile-index columns, then join on those for a tile-aligned two-raster `RS_MapAlgebra`. The same query works for single-tile inputs (as here) or for a DEM-sized scene that yields thousands of tiles per layer.

In [ ]:
rasters = (
    sedona.read.format("raster")
    .load(f"{WORK}/*.tif")
    .selectExpr("split(name, '\\\\.')[0] as layer", "x", "y", "rast")
)
rasters.createOrReplaceTempView("rasters")

composite = sedona.sql("""
    SELECT s.x, s.y,
           RS_MapAlgebra(
               s.rast, f.rast, 'D',
               'out[0] = 0.5 * rast0[0] + 0.5 * rast1[0];',
               -9999.0
           ) AS rast
    FROM (SELECT x, y, rast FROM rasters WHERE layer = 'slope') s
    JOIN (SELECT x, y, rast FROM rasters WHERE layer = 'fuel')  f
      ON s.x = f.x AND s.y = f.y
""")
composite.cache()
composite.createOrReplaceTempView("composite")
composite.selectExpr(
    "x",
    "y",
    "RS_BandPixelType(rast, 1) as dtype",
    "RS_Width(rast) as w",
    "RS_Height(rast) as h",
).show()

## 4. Synthesize building footprints and road network

Sixteen small square buildings on a 4×4 grid across the AOI, plus two roads (one east-west across the middle, one north-south down the centre). Buildings are 0.005° × 0.005° (~ 500 m × 500 m at this latitude); roads are simple `LINESTRING`s that bisect the AOI.

In [ ]:
from pyspark.sql import Row

xmin, ymin, xmax, ymax = AOI
step_x = (xmax - xmin) / 4
step_y = (ymax - ymin) / 4
half = 0.0025  # building half-width in degrees

building_rows = []
for ix in range(4):
    for iy in range(4):
        cx = xmin + (ix + 0.5) * step_x
        cy = ymin + (iy + 0.5) * step_y
        wkt = (
            f"POLYGON(({cx-half} {cy-half}, {cx+half} {cy-half}, "
            f"{cx+half} {cy+half}, {cx-half} {cy+half}, {cx-half} {cy-half}))"
        )
        building_rows.append(Row(bid=f"B{ix}{iy}", wkt=wkt))

buildings = sedona.createDataFrame(building_rows).selectExpr(
    "bid", "ST_GeomFromText(wkt) as geom"
)
buildings.createOrReplaceTempView("buildings")

mid_x = (xmin + xmax) / 2
mid_y = (ymin + ymax) / 2
road_rows = [
    Row(road_id="EW", wkt=f"LINESTRING({xmin} {mid_y}, {xmax} {mid_y})"),
    Row(road_id="NS", wkt=f"LINESTRING({mid_x} {ymin}, {mid_x} {ymax})"),
]
roads = sedona.createDataFrame(road_rows).selectExpr(
    "road_id", "ST_GeomFromText(wkt) as geom"
)
roads.createOrReplaceTempView("roads")

print(f"{buildings.count()} buildings, {roads.count()} roads")

## 5. Score every building

Two SQL passes:

- **Distance to nearest road** — group `(building × road)` cross product by building, take the `MIN(ST_DistanceSpheroid)`. Spheroidal distance returns metres regardless of the geometries' EPSG:4326 lon/lat units.
- **Risk score** — `mean composite risk × (1 + min(dist_to_road_km, 5) / 5)`. The clamp at 5 km caps the evacuation penalty so a single distant building doesn't dominate; the multiplicative form means a building only ranks high when it has *both* high terrain risk and poor road access.

In [ ]:
buildings_with_dist = sedona.sql("""
    SELECT b.bid, b.geom,
           MIN(ST_DistanceSpheroid(b.geom, r.geom)) AS dist_m
    FROM buildings b, roads r
    GROUP BY b.bid, b.geom
""")
buildings_with_dist.createOrReplaceTempView("buildings_with_dist")

scored = sedona.sql("""
    SELECT b.bid,
           ROUND(RS_ZonalStats(c.rast, b.geom, 'mean'), 4) AS mean_risk,
           ROUND(b.dist_m / 1000.0, 2) AS dist_km,
           ROUND(
               RS_ZonalStats(c.rast, b.geom, 'mean')
                 * (1.0 + LEAST(b.dist_m, 5000.0) / 5000.0),
               4
           ) AS risk_score,
           b.geom
    FROM buildings_with_dist b, composite c
    ORDER BY risk_score DESC
""")
scored.cache()
scored.createOrReplaceTempView("scored")
scored.select("bid", "mean_risk", "dist_km", "risk_score").show(16)

## 6. Persist the top-5 highest-risk buildings

Write top-5 to GeoParquet 1.1 — covering-bbox metadata and projjson CRS auto-populated — and read it back to confirm the round-trip.

In [ ]:
import shutil

out_path = f"{WORK}/top_risk_buildings.parquet"
if os.path.exists(out_path):
    shutil.rmtree(out_path)

top5 = sedona.sql("SELECT * FROM scored LIMIT 5")
top5.coalesce(1).write.format("geoparquet").save(out_path)

sedona.read.format("geoparquet").load(out_path).select(
    "bid", "mean_risk", "dist_km", "risk_score"
).show(truncate=False)

## 7. Visualize composite risk + scored buildings + roads

One matplotlib axes: composite risk raster as the basemap, building footprints filled by `risk_score` (red = high), roads overlaid as black lines, top-5 buildings labelled.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

scored_pdf = scored.selectExpr(
    "bid", "mean_risk", "dist_km", "risk_score", "ST_AsText(geom) as wkt"
).toPandas()
top_ids = set(scored_pdf.head(5)["bid"])

composite_arr = 0.5 * slope + 0.5 * fuel  # same expression used in SQL

fig, ax = plt.subplots(1, 1, figsize=(7, 7))
extent = (AOI[0], AOI[2], AOI[1], AOI[3])
ax.imshow(composite_arr, vmin=0, vmax=1, cmap="YlOrRd", extent=extent, alpha=0.7)

norm = Normalize(
    vmin=scored_pdf["risk_score"].min(), vmax=scored_pdf["risk_score"].max()
)
cmap = plt.colormaps.get_cmap("Reds")
for _, row in scored_pdf.iterrows():
    bbox = row["wkt"].split("((")[1].split("))")[0]
    pts = [tuple(float(x) for x in p.strip().split()) for p in bbox.split(",")]
    xs_b = [p[0] for p in pts]
    ys_b = [p[1] for p in pts]
    ax.fill(
        xs_b,
        ys_b,
        facecolor=cmap(norm(row["risk_score"])),
        edgecolor="black",
        linewidth=0.6,
    )
    if row["bid"] in top_ids:
        ax.text(
            sum(xs_b[:-1]) / 4,
            sum(ys_b[:-1]) / 4,
            row["bid"],
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
        )

ax.plot([AOI[0], AOI[2]], [(AOI[1] + AOI[3]) / 2] * 2, "k-", linewidth=2, alpha=0.6)
ax.plot([(AOI[0] + AOI[2]) / 2] * 2, [AOI[1], AOI[3]], "k-", linewidth=2, alpha=0.6)
ax.set_title("Composite risk + buildings (red = top-5 by score)")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
fig.colorbar(
    ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="risk_score", fraction=0.04
)
fig.tight_layout()
fig

## What just happened?

Six SQL passes turned synthetic terrain + footprint inputs into a ranked, persisted, plotted wildfire-risk report:

1. Loaded two synthesized GeoTIFFs with `sedona.read.format("raster")`.
2. Joined them on tile coordinates and combined via two-raster `RS_MapAlgebra` into a composite risk raster.
3. Built building polygons and road `LINESTRING`s as Spark DataFrames.
4. `MIN(ST_DistanceSpheroid)` over the building × road cross product gave each building's distance to its nearest road, in metres.
5. `RS_ZonalStats(composite, footprint, 'mean')` × evacuation factor produced a per-building risk score.
6. Wrote top-5 to GeoParquet 1.1, read back to verify, plotted everything on a single matplotlib axes.

The slope-east-fuel-north synthesis means buildings in the **north-east corner** carry the highest composite risk; the multiplicative evacuation factor then favours buildings that are also far from one of the two bisector roads. The ranking from the harness pass picks the corner buildings as expected — same SQL runs unchanged on a county-sized DEM + the OSM road network + Overture buildings, just with different inputs.